In [ ]:
# ==============================================================================
# ALL-IN-ONE SELF-SUFFICIENT EVALUATION & BENCHMARK SUITE FOR GOOGLE COLAB
# ==============================================================================
import os
import sys
import glob
from pathlib import Path

print("[*] Step 1/6: Checking GPU and Mounting Google Drive...")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

print("\n[*] Step 2/6: Installing Required Dependencies...")
!pip install -q ultralytics shapely scipy pandas matplotlib seaborn scikit-learn

print("\n[*] Step 3/6: Setting up project working directory...")
for candidate in ["/content/minor-project", ".", "/content/drive/MyDrive/minor-project"]:
    if os.path.exists(os.path.join(candidate, "scripts", "run_pipeline.py")):
        os.chdir(candidate)
        print(f"[✓] Active working directory: {os.getcwd()}")
        break

print("\n[*] Step 4/6: [Speedup Hack] Mirroring datasets to Colab Local Fast SSD...")
!mkdir -p /content/data
drive_data = "/content/drive/MyDrive/object-detection/data"
if os.path.exists(drive_data):
    !cp -r -u /content/drive/MyDrive/object-detection/data/* /content/data/ 2>/dev/null || true
    print("[✓] Fast local dataset mirror ready at /content/data")
else:
    print("[!] Drive data dir not detected; using existing local datasets.")

print("\n[*] Step 5/6: Executing Evaluation on Trained Checkpoints (post_eval + report)...")
!python scripts/run_pipeline.py \
    --steps post_eval report \
    --data-dir /content/data \
    --weights-dir /content/drive/MyDrive/object-detection/weights \
    --results-dir /content/drive/MyDrive/object-detection/results \
    --datasets visdrone codrone dota \
    --models std rvsa ars-detr oriented-former rio-detr rhino ao2-detr swin-obb lsknet yolo11-obb \
    --device cuda \
    --save-plots \
    --resume True \
    --run-name colab_evaluation_run

print("\n[*] Step 6/6: Rendering Results, Metrics Table, and Plots...")
import pandas as pd
from IPython.display import Image, display, Markdown

res_dirs = [
    "/content/drive/MyDrive/object-detection/results/colab_evaluation_run",
    "results/colab_evaluation_run"
]
active_res_dir = next((d for d in res_dirs if os.path.exists(d)), None)

if active_res_dir:
    csv_file = os.path.join(active_res_dir, "benchmark_summary.csv")
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        display_cols = [c for c in ["model", "dataset", "map50", "map75", "map50_95", "precision", "recall", "f1", "angle_mae", "fps"] if c in df.columns]
        print("\n========================== MODEL EVALUATION SUMMARY ==========================")
        display(df[display_cols].sort_values(by=["dataset", "map50"], ascending=[True, False]))

    comp_plot = os.path.join(active_res_dir, "plots", "model_benchmark_comparison.png")
    if os.path.exists(comp_plot):
        print("\n--- Model Performance Comparison (mAP50 & F1) ---")
        display(Image(comp_plot, width=800))

    rep_file = os.path.join(active_res_dir, "benchmark_report.md")
    if os.path.exists(rep_file):
        print("\n--- Benchmark Report Summary ---")
        with open(rep_file, "r", encoding="utf-8") as f:
            display(Markdown(f.read()))

    cm_plots = sorted(glob.glob(f"{active_res_dir}/plots/confusion_matrix_*.png"))
    if cm_plots:
        print(f"\n--- Displaying {len(cm_plots)} Confusion Matrices ---")
        for cm in cm_plots[:5]:
            print(f"\nPlot: {os.path.basename(cm)}")
            display(Image(cm, width=600))
else:
    print("[!] No results directory generated. Check training logs above for details.")
